In [95]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

In [96]:
#Loading Final Conflict Features Dataset
All_Conflict_Features = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/All_Conflict_Features.csv'))

#Loading Final News Topics Features Dataset
All_NewsTopic_Features = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/All_NewsTopic_Features.csv'))

#Loading Final Sociodemographic Features Dataset
All_SocDem_Features = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/All_SocDem_Features.csv'))

#Loading Final Election Features Dataset

In [97]:
print(All_Conflict_Features.shape)
print(All_Conflict_Features.columns)

(21996, 27)
Index(['ZCTA', 'month', 'admin2', 'fatalities', 'act_gov', 'act_pol',
       'act_nat', 'act_rel', 'act_edu', 'act_wor', 'act_iss', 'act_rwm',
       'act_oth', 'set_peaceful_prot', 'set_other', 'set_prot_int', 'set_mob',
       'set_violent_prot', 'set_loot_destr', 'unions', 'environment',
       'gun-violence', 'healthcare', 'event_count', 'neighbor_conflicts',
       'event_count_prev_year', 'event_count_prev_month'],
      dtype='object')


In [98]:
print(All_NewsTopic_Features.shape)

(21996, 16)


In [99]:
print(All_SocDem_Features.shape)

(21996, 174)


In [100]:
#Ensure that datasets are all in the same order + add index --> not really necessary as this has been done before but a safety measure 
#Sorting all ZIPs in ascending order 
All_Conflict_Features = All_Conflict_Features.sort_values(by=['ZCTA', 'month'], ascending=[True, True])
All_SocDem_Features = All_SocDem_Features.sort_values(by=['ZCTA', 'Month'], ascending=[True, True])

#Resetting index 
All_Conflict_Features.reset_index(drop=True, inplace=True)
All_SocDem_Features.reset_index(drop=True, inplace=True)
All_NewsTopic_Features.reset_index(drop=True, inplace=True)

In [101]:
#Merging on index: Merging conflict features with socio demographic features
Final_Dataset_Lag = All_Conflict_Features.merge(All_SocDem_Features, left_index=True, right_index=True, how='left', suffixes=('_conf', '_sd'))

#Merging conflicts in prev year 
Final_Dataset_Lag = Final_Dataset_Lag.merge(All_NewsTopic_Features, left_index=True, right_index=True, how='left', suffixes=('', '_news'))

In [102]:
#Checking all col names
print(Final_Dataset_Lag.columns)

Index(['ZCTA_conf', 'month', 'admin2', 'fatalities', 'act_gov', 'act_pol',
       'act_nat', 'act_rel', 'act_edu', 'act_wor',
       ...
       'topic5', 'topic6', 'topic7', 'topic8', 'topic9', 'topic10', 'topic11',
       'topic12', 'topic13', 'topic14'],
      dtype='object', length=217)


In [103]:
#Dropping duplicate cols
Final_Dataset_Lag = Final_Dataset_Lag.drop(['ZCTA_sd', 'Month'], axis=1)

#Renaming cols
Final_Dataset_Lag = Final_Dataset_Lag.rename(columns={'ZCTA_conf': 'ZCTA', 'Name of ZIP Code': 'Post Office City Name'})

print(Final_Dataset_Lag.columns)


Index(['ZCTA', 'month', 'admin2', 'fatalities', 'act_gov', 'act_pol',
       'act_nat', 'act_rel', 'act_edu', 'act_wor',
       ...
       'topic5', 'topic6', 'topic7', 'topic8', 'topic9', 'topic10', 'topic11',
       'topic12', 'topic13', 'topic14'],
      dtype='object', length=215)


In [104]:
print(Final_Dataset_Lag.shape)

(21996, 215)


In [105]:
#save as csv 
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset')
output_filename_1 = "Final_Dataset_Lag.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

Final_Dataset_Lag.to_csv(output_path_1, index=False)

In [106]:
#Drop lags 

Final_Dataset_noLag = Final_Dataset_Lag.drop(['event_count_prev_year', 'event_count_prev_month'], axis=1)
print(Final_Dataset_noLag.shape)

(21996, 213)


In [107]:
#save dataset without lag as csv 
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/d_Final_Dataset')
output_filename_1 = "Final_Dataset_noLag.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

Final_Dataset_noLag.to_csv(output_path_1, index=False)